##### - Use this notebook to search for coincident (non-cloudy) Landsat8 and Sentinel-2 images
##### - The script exports respective Landsat8 and Sentinel-2 footprints for coincident images as shapefiles
##### - The footprints are indexed by observation date. 
##### - Use the following script '2-calculate-overlapping-footprints' to find dates and corresponding footprints with the highest portion of non-cloudy Sentinel-2/Landsat8 overlap. 

## 1.0 Libraries and directories

In [1]:
import ee 
import geemap
import geopandas as gpd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_name = 'YKD_sub1'
cloud_threshold = 10

""" 
To select coincident images from a specific seasons, 
make list of timeframes with begining and end.
"""
season_start = '05-15' # 'MM-DD'
season_stop = '09-15'
year_start = 2015 # YYYY
year_stop = 2025
if year_start != year_stop:
    year_range = range(year_start, year_stop)
else:
    year_range = [year_start]

timeframes = []
for year in year_range:
    begin = str(year) + '-' + season_start
    end = str(year) + '-' + season_stop
    tf = (begin, end)
    timeframes.append(tf)

Exception: Problem requesting tokens. Please try again.  HTTP Error 400: Bad Request b'{\n  "error": "invalid_request",\n  "error_description": "Missing required parameter: code"\n}'

## 2.0 Convert ROI to Earth Engine Polygon

In [8]:
roi = gpd.read_file(f'./data/roi_shapes/{roi_name}_shape.shp')
#Just get the first geometry
geom = roi.geometry.iloc[0] 
coords = list(geom.exterior.coords)
coords_list = [[x, y] for x, y in coords]
roi = ee.Geometry.Polygon(coords_list)

## 3.0 Functions to search images and calculate footprints

In [9]:
def find_img_pairs(roi, start, end, cloud_threshold):
    """
    Searches for images in the region and timeframe with a desired cloud threshold
    Returns a paired collection with images from an inner join by date
    """
    s2_string = 'COPERNICUS/S2' # L1C data
    ls8_string = 'LANDSAT/LC08/C02/T1_L2'
    # Produces image collection for all images within the roi and date range
    s2_col = (
        ee.ImageCollection(s2_string) 
        .filterDate(start, end)
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
    )

    ls8_col = (
        ee.ImageCollection(ls8_string)
        .filterDate(start, end)
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUD_COVER', cloud_threshold))
    )

    def add_date_for_join(img):
        """Formatts the date consistently to join image collections"""
        return img.set('formatted_date', img.date().format('YYYYMMdd'))

    # Join image collections by date into a paired collection
    s2_col = s2_col.map(add_date_for_join)
    ls8_col = ls8_col.map(add_date_for_join)
    date_filter = ee.Filter.equals(leftField='formatted_date', rightField='formatted_date')
    inner_join = ee.Join.inner()
    paired_collection = inner_join.apply(s2_col, ls8_col, date_filter)

    return paired_collection

def separate_satellites(paired_fc):
    """
    Makes two sepperate collections from the combined paired collections.
    This allows you to calculate each satellite's respective footprint on the coincindent date. 
    """
    s2_images = paired_fc.map(
        lambda feature: (
            ee.Image(feature.get('primary'))
                .set('formatted_date', ee.Image(feature.get('primary')).get('formatted_date'))
        )
    )
    s2_col = ee.ImageCollection(s2_images)

    ls8_images = paired_fc.map(
        lambda feature: (
            ee.Image(feature.get('secondary'))
                .set('formatted_date', ee.Image(feature.get('secondary')).get('formatted_date'))
        )
    )
    ls8_col = ee.ImageCollection(ls8_images)

    return s2_col, ls8_col

def mosiac_attrs_by_date(img_col, roi, satellite):
    """
    Uses the image collection of a given satellite (Sentinel-2 or Landsat8),
    Returns polygon feature collections with each date's corresponding non-cloudy footprint
    """

    # Using only one band to prevent wonky segmentation with the reduceToVectors() method
    if satellite == 'Sentinel-2':
        scale = 10
        select_band = 'B8'
    else:
        scale = 30
        select_band = 'SR_B5'
    
    distinct_dates = img_col.aggregate_array('formatted_date').distinct()

    def date_mosaic(date_str):
        """
        Makes an image mosiac for each date and returns a polygon with mosaic's footprint. 
        """
        date_str = ee.String(date_str)
        date_imgs = img_col.filter(ee.Filter.eq('formatted_date', date_str))
        mosaic = (date_imgs.mosaic()
                  .set('formatted_date', date_str)
                  .clip(roi)
                  .select(select_band))
        
        data_mask = mosaic.gt(0)
        # Fill random small holes in the data mask
        radius = 5 # pixels
        kernel = ee.Kernel.circle(radius, 'pixels')
        filled_data_mask = data_mask.focal_max(kernel=kernel).focal_min(kernel=kernel)


        polygon_boundaries = filled_data_mask.reduceToVectors(
            #geometry=roi,
            geometryType='polygon',
            scale=scale,
            maxPixels=1e13,
            eightConnected=True
        )

        # Take polygons with a large enough area, there's some random small polygons due to wonky segmentation
        polygons_with_area = polygon_boundaries.map(lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1)
        }))
        large_enough = polygons_with_area.filter(ee.Filter.gt('area_m2', 100000))
        # Make polygons into a single geometry
        combined = large_enough.geometry(maxError=1)

        final_feature = ee.Feature(combined).set('formatted_date', date_str)
        
        return final_feature

    # Feature collection with each date's footprint
    footprints_fc = ee.FeatureCollection(distinct_dates.map(date_mosaic))
   
    return footprints_fc
    

## 4.0 Run the functions and produce footprints

In [10]:
all_imgs_list = []

for tf in timeframes:
    start = tf[0]
    end = tf[1]
    #print(f'---------  Observations for {tf}  ----------')
    paired = find_img_pairs(roi, start, end, cloud_threshold=cloud_threshold)
    all_imgs_list.append(paired)

all_imgs = all_imgs_list[0]
for i in range(1, len(all_imgs_list)):
    all_imgs = all_imgs.merge(all_imgs_list[i])

s2, ls8 = separate_satellites(paired_fc=all_imgs)
footprints_s2 = mosiac_attrs_by_date(s2, roi=roi, satellite='Sentinel-2')
footprints_ls8 = mosiac_attrs_by_date(ls8, roi=roi, satellite='Landsat8')

#print(s2.size().getInfo())

## 4.0 Export the footprints to Drive as shapefiles

In [11]:
task_s2 = ee.batch.Export.table.toDrive(
    collection=footprints_s2,
    description=f'footprints_s2_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',
    folder='s2_roi_img_footprints',            
    fileNamePrefix=f'footprints_s2_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',      
    fileFormat='SHP',
)
task_s2.start()

In [12]:
task_ls8 = ee.batch.Export.table.toDrive(
    collection=footprints_ls8,
    description=f'footprints_ls8_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',
    folder='ls8_roi_img_footprints',            
    fileNamePrefix=f'footprints_ls8_roi_{roi_name}_years_{year_start}_{year_stop - 1}_dates_{season_start}_{season_stop}',      
    fileFormat='SHP',
)
task_ls8.start()